# SkillGap AI Pro - Colab Deployment

Full-stack deployment: Flask API + React frontend with 25 roles, 68K courses, ML gap analyzer.

## Instructions
1. Run cells sequentially 1-9
2. Edit the REPO_URL and DB_URL params in step 2
3. Frontend accessible via Colab proxy on port 8501

In [ ]:
# @title 1. Mount Google Drive (for DB persistence)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# @title 2. Clone Repo + Set DB Source
import os, shutil

REPO_URL = "https://github.com/your-username/SkillGapAI_Project"  # @param {type:"string"}
DB_URL = ""  # @param {type:"string"} (URL to pre-built skill_progress.db, optional)
PROJECT = "/content/SkillGapAI_Project"

if not os.path.exists(PROJECT):
    !git clone {REPO_URL} {PROJECT}
else:
    %cd {PROJECT}
    !git pull
%cd {PROJECT}

In [ ]:
# @title 3. Install Python Dependencies
!pip install flask flask-cors pandas numpy scikit-learn joblib sentence-transformers 2>&1 | tail -3
!pip install pyngrok 2>&1 | tail -3

In [ ]:
# @title 4. Install Node.js + Build React Frontend
import os
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - 2>&1 | tail -1
!apt-get install -y nodejs 2>&1 | tail -3
FRONTEND = "/content/SkillGapAI_Project/frontend"
%cd {FRONTEND}
!npm install 2>&1 | tail -3
!npm run build 2>&1 | tail -5
%cd /content/SkillGapAI_Project

In [ ]:
# @title 5. Setup Database
import sqlite3, urllib.request, zipfile, os

DB = "data/skill_progress.db"

def db_ready():
    if not os.path.exists(DB):
        return False
    conn = sqlite3.connect(DB)
    tables = [t[0] for t in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")]
    conn.close()
    return "courses" in tables and "roles" in tables

if db_ready():
    print("Database ready.")
elif DB_URL:
    print("Downloading pre-built database...")
    urllib.request.urlretrieve(DB_URL, "skill_progress.db.zip")
    with zipfile.ZipFile("skill_progress.db.zip", "r") as z:
        z.extractall("data/")
    print("Database downloaded and extracted.")
else:
    print("No DB found and no DB_URL provided.")
    print("TIP: Upload skill_progress.db to Google Drive, then:")
    print('  !cp "/content/drive/MyDrive/skill_progress.db" data/')
    print("Or place CSV exports in data/exports/ and run: !python import_course_data.py")

In [ ]:
# @title 6. Start Flask Backend
import subprocess, sys, time
flask_proc = subprocess.Popen([sys.executable, "app.py"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(4)
print(f"Flask PID: {flask_proc.pid} | http://127.0.0.1:5000/api")

# Quick health check
import requests
try:
    r = requests.get("http://127.0.0.1:5000/api/roles", timeout=5)
    print(f"Roles: {len(r.json())} loaded")
except:
    print("WARNING: Flask not responding yet (models may still loading)")

In [ ]:
# @title 7. Ngrok Tunnel (optional, port 5000 serves both API + Frontend)
from pyngrok import ngrok
NGROK_TOKEN = ""  # @param {type:"string"}
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
    url = ngrok.connect(5000, "http")
    print(f"Public URL: {url}")
    print("Opens frontend at /api/* for API")
else:
    print("No ngrok token - local only")

In [ ]:
# @title 8. Frontend served by Flask (skip separate Vite server)
print("Flask on port 5000 serves built React frontend + API.")
print(f"Open: http://localhost:5000 (or ngrok URL from cell 7)")
print("NOTE: Run npm run build after any frontend changes.")

In [ ]:
# @title 9. Test with Sample Data
import requests
payload = {
    "target_role": "Data Scientist",
    "scores": {"Logical": 750, "Quant": 800, "English": 700, "ComputerProgramming": 650, "Domain": 600},
    "selected_skills": ["Python", "Machine Learning", "SQL", "Statistics", "Deep Learning"],
    "resume_text": "Data scientist with Python, ML, SQL, Deep Learning, Statistics. Built predictive models."
}
r = requests.post("http://127.0.0.1:5000/api/analyze", json=payload)
d = r.json()
print(f"Score: {d.get('final_readiness_score','N/A')}% | Ready: {d.get('is_job_ready','N/A')}")
print(f"Missing: {d.get('missing_skills',[])[:5]}")
print(f"AI Role: {d.get('ai_role','N/A')}")

In [ ]:
# @title Stop All
if "flask_proc" in dir(): flask_proc.terminate()
try:
    ngrok.kill()
except:
    pass
print("Stopped.")